# توسعه پروژه NER فارسی با MAD-X

## انتقال Zero-shot وظیفه NER از انگلیسی به فارسی

در این آزمایش، Language Adapter فارسی با متن بدون برچسب آموزش داده می‌شود، Task Adapter وظیفه NER را از داده انگلیسی یاد می‌گیرد و سپس ترکیب آن‌ها بدون آموزش NER فارسی روی داده فارسی ارزیابی می‌شود.

## خلاصه آزمایش

- مدل پایه: `xlm-roberta-base`
- داده: WikiANN انگلیسی و فارسی
- روش: MAD-X با Language Adapter و Task Adapter
- آموزش تسک NER: فقط با داده برچسب‌دار انگلیسی
- ارزیابی هدف: فارسی به‌صورت Cross-lingual Zero-shot
- F1 تست انگلیسی: **0.7686**
- F1 فارسی Zero-shot: **0.7738**

> این نوت‌بوک یک پیاده‌سازی سبک و آموزشی از MAD-X است. Language Adapterها با ۱۸ هزار جمله کوتاه WikiANN آموزش داده شده‌اند، نه با پیکره‌های بزرگ Wikipedia.


## راهنمای اجرا

نوت‌بوک را از بالا به پایین اجرا کنید. پس از اجرای سلول نصب کتابخانه‌ها، یک‌بار از مسیر  
`Runtime → Restart session`  
محیط Colab را ری‌استارت کنید و سپس اجرا را از سلول بررسی نسخه‌ها ادامه دهید.

اجرای کامل شامل آموزش دو Language Adapter و یک Task Adapter است و روی Tesla T4 حدود ۴۰ دقیقه زمان می‌برد.


## 1. آماده‌سازی محیط


In [1]:
# نصب نسخه‌های سازگار کتابخانه‌ها

%pip install -q \
    "transformers==4.57.6" \
    "adapters==1.3.0" \
    "datasets==4.0.0" \
    "evaluate==0.4.6" \
    "seqeval==1.2.2" \
    accelerate \
    sentencepiece


### توقف کوتاه برای Restart

بعد از پایان نصب، Runtime را یک‌بار Restart کنید. سپس از سلول بعدی ادامه دهید و سلول نصب را دوباره اجرا نکنید.


In [2]:
# بررسی نصب صحیح کتابخانه‌ها پس از Restart

import torch
import transformers
import datasets
import evaluate
from importlib.metadata import version
from adapters import AutoAdapterModel

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Adapters:", version("adapters"))
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU فعال نیست")
print("AutoAdapterModel با موفقیت بارگذاری شد.")

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Adapters: 1.3.0
Datasets: 4.0.0
Evaluate: 0.4.6
GPU: Tesla T4
AutoAdapterModel با موفقیت بارگذاری شد.


In [3]:
# ساخت پوشه‌های پروژه MAD-X در Google Drive

from pathlib import Path
from google.colab import drive
import random
import numpy as np
import torch

drive.mount("/content/drive")

# مسیر اصلی پروژه
PROJECT_DIR = Path(
    "/content/drive/MyDrive/persian-ner-madx"
)

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
LANG_ADAPTER_DIR = OUTPUT_DIR / "fa_language_adapter"
EN_LANG_ADAPTER_DIR = OUTPUT_DIR / "en_language_adapter"
TASK_ADAPTER_DIR = OUTPUT_DIR / "en_ner_task_adapter"
RESULTS_DIR = PROJECT_DIR / "results"

# ساخت پوشه‌ها
for path in [
    PROJECT_DIR,
    DATA_DIR,
    OUTPUT_DIR,
    LANG_ADAPTER_DIR,
    EN_LANG_ADAPTER_DIR,
    TASK_ADAPTER_DIR,
    RESULTS_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

# ثابت‌کردن Seed برای تکرارپذیری
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("مسیر پروژه:", PROJECT_DIR)
print("پوشه‌ها با موفقیت ساخته شدند.")

Mounted at /content/drive
مسیر پروژه: /content/drive/MyDrive/persian-ner-madx
پوشه‌ها با موفقیت ساخته شدند.


## 2. بارگذاری داده و تعریف برچسب‌ها


In [4]:
# بارگذاری دیتاست WikiANN برای انگلیسی و فارسی

from datasets import load_dataset

english_dataset = load_dataset(
    "unimelb-nlp/wikiann",
    "en"
)

persian_dataset = load_dataset(
    "unimelb-nlp/wikiann",
    "fa"
)

print("English dataset:")
print(english_dataset)

print("\nPersian dataset:")
print(persian_dataset)

print("\nنمونه انگلیسی:")
print(english_dataset["train"][0])

print("\nنمونه فارسی:")
print(persian_dataset["train"][0])

print("\nساختار برچسب‌ها:")
print(english_dataset["train"].features)

warnings.warn(


English dataset:
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

Persian dataset:
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

نمونه انگلیسی:
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R

In [5]:
# بررسی یکسان‌بودن برچسب‌های NER در انگلیسی و فارسی

english_label_names = (
    english_dataset["train"]
    .features["ner_tags"]
    .feature.names
)

persian_label_names = (
    persian_dataset["train"]
    .features["ner_tags"]
    .feature.names
)

print("برچسب‌های انگلیسی:")
print(english_label_names)

print("\nبرچسب‌های فارسی:")
print(persian_label_names)

assert english_label_names == persian_label_names, (
    "ترتیب برچسب‌های دو زبان یکسان نیست."
)

label_names = english_label_names
num_labels = len(label_names)

id2label = {
    index: label
    for index, label in enumerate(label_names)
}

label2id = {
    label: index
    for index, label in enumerate(label_names)
}

print("\nتعداد برچسب‌ها:", num_labels)
print("نگاشت ID به برچسب:", id2label)
print("\nبرچسب‌های دو زبان کاملاً یکسان هستند.")

برچسب‌های انگلیسی:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

برچسب‌های فارسی:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

تعداد برچسب‌ها: 7
نگاشت ID به برچسب: {0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}

برچسب‌های دو زبان کاملاً یکسان هستند.


## 3. آموزش Language Adapter فارسی


In [6]:
# ساخت دیتاست فارسی بدون برچسب برای آموزش Language Adapter

def create_unlabeled_text(example):
    return {
        "text": " ".join(example["tokens"])
    }

# فقط از بخش train فارسی استفاده می‌کنیم
fa_unlabeled = persian_dataset["train"].map(
    create_unlabeled_text,
    remove_columns=persian_dataset["train"].column_names
)

# تقسیم داخلی برای آموزش و ارزیابی Masked Language Modeling
fa_mlm_dataset = fa_unlabeled.train_test_split(
    test_size=0.10,
    seed=SEED
)

# تغییر نام test داخلی به validation
fa_mlm_dataset["validation"] = fa_mlm_dataset.pop("test")

# ذخیره در Google Drive
FA_MLM_RAW_DIR = DATA_DIR / "fa_mlm_raw"

fa_mlm_dataset.save_to_disk(
    str(FA_MLM_RAW_DIR)
)

print(fa_mlm_dataset)

print("\nنمونه متن فارسی بدون برچسب:")
print(fa_mlm_dataset["train"][0])

print("\nتعداد نمونه‌های آموزش:", len(fa_mlm_dataset["train"]))
print("تعداد نمونه‌های validation:", len(fa_mlm_dataset["validation"]))

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 18000
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 2000
    })
})

نمونه متن فارسی بدون برچسب:
{'text': 'تغییر_مسیر ابوالوفا محمد بوزجانی'}

تعداد نمونه‌های آموزش: 18000
تعداد نمونه‌های validation: 2000


In [7]:
# توکن‌سازی متن‌های فارسی برای آموزش Language Adapter

from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
MAX_MLM_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

def tokenize_fa_text(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_MLM_LENGTH,
        return_special_tokens_mask=True
    )

fa_mlm_tokenized = fa_mlm_dataset.map(
    tokenize_fa_text,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing Persian MLM dataset"
)

# ذخیره نسخه توکنایزشده
FA_MLM_TOKENIZED_DIR = DATA_DIR / "fa_mlm_tokenized"

fa_mlm_tokenized.save_to_disk(
    str(FA_MLM_TOKENIZED_DIR)
)

print(fa_mlm_tokenized)

print("\nنمونه توکنایزشده:")
print(fa_mlm_tokenized["train"][0])

print("\nتوکن‌های نمونه:")
print(
    tokenizer.convert_ids_to_tokens(
        fa_mlm_tokenized["train"][0]["input_ids"]
    )
)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 18000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 2000
    })
})

نمونه توکنایزشده:
{'input_ids': [0, 25012, 454, 32634, 7282, 28754, 1282, 92564, 2977, 676, 14800, 20717, 140, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'special_tokens_mask': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]}

توکن‌های نمونه:
['<s>', '▁تغییر', '_', 'مس', 'یر', '▁ابو', 'ال', 'وفا', '▁محمد', '▁ب', 'وز', 'جان', 'ی', '</s>']


In [8]:
# بارگذاری مدل پایه و بررسی Prediction Headهای موجود

from adapters import AutoAdapterModel

model = AutoAdapterModel.from_pretrained(
    MODEL_NAME
)

print("کلاس مدل:", type(model).__name__)
print("Prediction headهای موجود:", list(model.heads.keys()))
print("\nتنظیمات Adapterهای مدل:")
print(model.adapters_config)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("\nتعداد کل پارامترها:", f"{total_params:,}")

کلاس مدل: XLMRobertaAdapterModel
Prediction headهای موجود: ['default']

تنظیمات Adapterهای مدل:

تعداد کل پارامترها: 278,885,778


In [9]:
# این سلول بررسی می‌کند که Head پیش‌فرض مدل برای Masked Language Modeling
# قابل استفاده است و می‌تواند Loss و Logits معتبر تولید کند.

import torch
from transformers import DataCollatorForLanguageModeling

# انتقال مدل به GPU
model = model.to("cuda")

# نمایش ساختار Head پیش‌فرض
print("ساختار Head پیش‌فرض:")
print(model.heads["default"])

# این Collator حدود ۱۵ درصد توکن‌ها را به‌صورت تصادفی ماسک می‌کند
mlm_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# انتخاب دو نمونه کوچک فقط برای آزمایش عملکرد Head
sample_features = [
    fa_mlm_tokenized["train"][0],
    fa_mlm_tokenized["train"][1]
]

# ساخت یک Batch شامل input_ids، attention_mask و labels
test_batch = mlm_data_collator(sample_features)

# انتقال داده‌های آزمایشی به GPU
test_batch = {
    key: value.to("cuda")
    for key, value in test_batch.items()
}

# اجرای آزمایشی مدل بدون محاسبه Gradient
with torch.no_grad():
    test_outputs = model(
        **test_batch,
        head="default"
    )

print("\nLoss آزمایشی:", test_outputs.loss.item())
print("شکل Logits:", tuple(test_outputs.logits.shape))
print("اندازه واژگان مدل:", tokenizer.vocab_size)

ساختار Head پیش‌فرض:
BertStyleMaskedLMHead(
  (0): Linear(in_features=768, out_features=768, bias=True)
  (1): Activation_Function_Class(
    (f): GELUActivation()
  )
  (2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (3): Linear(in_features=768, out_features=250002, bias=True)
)

Loss آزمایشی: 7.089354038238525
شکل Logits: (2, 14, 250002)
اندازه واژگان مدل: 250002


In [10]:
# این سلول Language Adapter فارسی مخصوص MAD-X را به مدل اضافه می‌کند،
# مدل اصلی را ثابت نگه می‌دارد و فقط پارامترهای Adapter فارسی را قابل‌آموزش می‌کند.

from adapters import SeqBnInvConfig

# نامی که برای ذخیره و بازیابی Adapter فارسی استفاده می‌کنیم
LANG_ADAPTER_NAME = "fa_language"

# تنظیمات Language Adapter:
# reduction_factor=2 یعنی اندازه فضای میانی Adapter برابر 768/2 است.
# inv_adapter="nice" بخش وارون‌پذیر موردنیاز MAD-X را فعال می‌کند.
fa_language_config = SeqBnInvConfig(
    reduction_factor=2,
    non_linearity="relu",
    inv_adapter="nice",
    inv_adapter_reduction_factor=2
)

# افزودن Language Adapter فارسی به مدل
model.add_adapter(
    LANG_ADAPTER_NAME,
    config=fa_language_config
)

# ثابت‌کردن مدل پایه و قابل‌آموزش‌کردن فقط Adapter فارسی
model.train_adapter(LANG_ADAPTER_NAME)

# فعال‌کردن Adapter فارسی در مسیر عبور داده
model.set_active_adapters(LANG_ADAPTER_NAME)

# فعال‌کردن Head مربوط به Masked Language Modeling
model.active_head = "default"

# انتقال مدل به GPU
model = model.to("cuda")

# شمارش پارامترهای کل و قابل‌آموزش
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

trainable_percent = (
    100 * trainable_params / total_params
)

print("Adapter فعال:", model.active_adapters)
print("Head فعال:", model.active_head)

print(
    "تعداد کل پارامترها:",
    f"{total_params:,}"
)

print(
    "پارامترهای قابل‌آموزش:",
    f"{trainable_params:,}"
)

print(
    "درصد پارامترهای قابل‌آموزش:",
    f"{trainable_percent:.4f}%"
)

print("\nLanguage Adapter فارسی با موفقیت ساخته و فعال شد.")

Adapter فعال: Stack[fa_language]
Head فعال: default
تعداد کل پارامترها: 286,273,554
پارامترهای قابل‌آموزش: 7,979,904
درصد پارامترهای قابل‌آموزش: 2.7875%

Language Adapter فارسی با موفقیت ساخته و فعال شد.


In [11]:
# این سلول تنظیمات آموزش Language Adapter فارسی را تعریف می‌کند
# و یک AdapterTrainer می‌سازد؛ در این مرحله هنوز آموزش آغاز نمی‌شود.

from transformers import TrainingArguments
from adapters import AdapterTrainer

# مسیر ذخیره checkpointهای آموزش Language Adapter فارسی
FA_LANG_TRAIN_DIR = OUTPUT_DIR / "fa_language_training"

# تنظیمات آموزش
fa_language_training_args = TrainingArguments(
    output_dir=str(FA_LANG_TRAIN_DIR),

    # تنظیمات اصلی آموزش
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    # ارزیابی و ذخیره در پایان هر epoch
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    # انتخاب بهترین checkpoint بر اساس کمترین validation loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # تنظیمات حافظه و سرعت
    fp16=True,
    weight_decay=0.01,

    # نگه‌داشتن فقط دو checkpoint آخر
    save_total_limit=2,

    # غیرفعال‌کردن سرویس‌های گزارش‌دهی خارجی
    report_to="none",

    # قابل‌تکرار بودن آزمایش
    seed=SEED,
    data_seed=SEED
)

# ساخت Trainer مخصوص آموزش Adapter
fa_language_trainer = AdapterTrainer(
    model=model,
    args=fa_language_training_args,
    train_dataset=fa_mlm_tokenized["train"],
    eval_dataset=fa_mlm_tokenized["validation"],
    data_collator=mlm_data_collator,
    processing_class=tokenizer
)

# محاسبه تعداد تقریبی گام‌های آموزش
effective_batch_size = (
    fa_language_training_args.per_device_train_batch_size
    * fa_language_training_args.gradient_accumulation_steps
)

steps_per_epoch = (
    len(fa_mlm_tokenized["train"])
    // effective_batch_size
)

print("Trainer با موفقیت ساخته شد.")
print("Batch مؤثر:", effective_batch_size)
print("تعداد تقریبی گام در هر epoch:", steps_per_epoch)
print("تعداد تقریبی کل گام‌ها:", steps_per_epoch * 3)
print("مسیر ذخیره checkpointها:", FA_LANG_TRAIN_DIR)

Trainer با موفقیت ساخته شد.
Batch مؤثر: 32
تعداد تقریبی گام در هر epoch: 562
تعداد تقریبی کل گام‌ها: 1686
مسیر ذخیره checkpointها: /content/drive/MyDrive/persian-ner-madx/outputs/fa_language_training


In [12]:
# این سلول Language Adapter فارسی را با روش Masked Language Modeling آموزش می‌دهد
# و زمان آموزش و بیشترین حافظه مصرف‌شده GPU را ثبت می‌کند.

import time
import torch

# آزادکردن حافظه‌های بلااستفاده GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# ثبت زمان شروع آموزش
start_time = time.perf_counter()

# شروع آموزش Language Adapter فارسی
fa_language_train_result = fa_language_trainer.train()

# محاسبه زمان کل آموزش
training_time_seconds = time.perf_counter() - start_time
training_time_minutes = training_time_seconds / 60

# محاسبه بیشترین حافظه مصرف‌شده GPU
peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

print("\nآموزش Language Adapter فارسی تمام شد.")
print(
    "زمان آموزش:",
    f"{training_time_minutes:.2f} دقیقه"
)
print(
    "بیشترین حافظه GPU:",
    f"{peak_gpu_memory_gb:.2f} GB"
)
print(
    "Training loss:",
    fa_language_train_result.training_loss
)

Epoch,Training Loss,Validation Loss
1,4.464900,2.064116
2,4.398600,2.007292
3,4.106500,1.903175


آموزش Language Adapter فارسی تمام شد.
زمان آموزش: 13.86 دقیقه
بیشترین حافظه GPU: 7.13 GB
Training loss: 4.526120553431672


In [13]:
# این سلول بهترین Language Adapter فارسی را ذخیره می‌کند،
# عملکرد نهایی آن را روی validation می‌سنجد و خلاصه نتایج را در فایل JSON می‌نویسد.

import json
import math

# ارزیابی بهترین checkpoint روی داده validation
fa_language_eval_results = fa_language_trainer.evaluate()

validation_loss = fa_language_eval_results["eval_loss"]
perplexity = math.exp(validation_loss)

# ذخیره فقط Language Adapter فارسی
model.save_adapter(
    str(LANG_ADAPTER_DIR),
    LANG_ADAPTER_NAME
)

# ذخیره Tokenizer برای استفاده‌های بعدی
TOKENIZER_DIR = OUTPUT_DIR / "tokenizer"

tokenizer.save_pretrained(
    str(TOKENIZER_DIR)
)

# ساخت خلاصه نتایج آموزش
fa_language_summary = {
    "adapter_name": LANG_ADAPTER_NAME,
    "training_method": "Masked Language Modeling",
    "train_samples": len(fa_mlm_tokenized["train"]),
    "validation_samples": len(fa_mlm_tokenized["validation"]),
    "epochs": 3,
    "learning_rate": 1e-4,
    "effective_batch_size": 32,
    "validation_loss": validation_loss,
    "perplexity": perplexity,
    "training_loss": fa_language_train_result.training_loss,
    "training_time_minutes": training_time_minutes,
    "peak_gpu_memory_gb": peak_gpu_memory_gb,
    "trainable_parameters": trainable_params,
    "trainable_percent": trainable_percent
}

# ذخیره نتایج در فایل JSON
FA_LANGUAGE_RESULTS_FILE = (
    RESULTS_DIR / "fa_language_adapter_results.json"
)

with open(
    FA_LANGUAGE_RESULTS_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        fa_language_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Language Adapter فارسی ذخیره شد.")
print("مسیر Adapter:", LANG_ADAPTER_DIR)
print("Validation Loss:", f"{validation_loss:.4f}")
print("Perplexity:", f"{perplexity:.4f}")
print("مسیر نتایج:", FA_LANGUAGE_RESULTS_FILE)

Language Adapter فارسی ذخیره شد.
مسیر Adapter: /content/drive/MyDrive/persian-ner-madx/outputs/fa_language_adapter
Validation Loss: 2.0064
Perplexity: 7.4368
مسیر نتایج: /content/drive/MyDrive/persian-ner-madx/results/fa_language_adapter_results.json


## 4. آموزش Language Adapter انگلیسی


In [14]:
# این سلول داده‌های انگلیسی WikiANN را به متن بدون برچسب تبدیل می‌کند
# و آن‌ها را برای آموزش Language Adapter انگلیسی توکن‌سازی می‌کند.
# ستون ner_tags حذف می‌شود؛ بنابراین در این مرحله هیچ NERای آموزش داده نمی‌شود.

from datasets import DatasetDict

# تبدیل فهرست کلمه‌ها به یک جمله انگلیسی
def create_english_unlabeled_text(example):
    return {
        "text": " ".join(example["tokens"])
    }

# فقط بخش train انگلیسی استفاده می‌شود
en_unlabeled = english_dataset["train"].map(
    create_english_unlabeled_text,
    remove_columns=english_dataset["train"].column_names,
    desc="Creating unlabeled English text"
)

# تقسیم به آموزش و validation با همان seed قبلی
en_mlm_dataset = en_unlabeled.train_test_split(
    test_size=0.10,
    seed=SEED
)

en_mlm_dataset["validation"] = en_mlm_dataset.pop("test")

# تابع توکن‌سازی برای Masked Language Modeling
def tokenize_english_text(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_MLM_LENGTH,
        return_special_tokens_mask=True
    )

# توکن‌سازی داده‌های انگلیسی
en_mlm_tokenized = en_mlm_dataset.map(
    tokenize_english_text,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing English MLM dataset"
)

# ذخیره نسخه کوچک توکن‌سازی‌شده در Drive
EN_MLM_TOKENIZED_DIR = DATA_DIR / "en_mlm_tokenized"

en_mlm_tokenized.save_to_disk(
    str(EN_MLM_TOKENIZED_DIR)
)

print(en_mlm_tokenized)

print("\nنمونه توکنایزشده انگلیسی:")
print(en_mlm_tokenized["train"][0])

print("\nتوکن‌های نمونه:")
print(
    tokenizer.convert_ids_to_tokens(
        en_mlm_tokenized["train"][0]["input_ids"]
    )
)

print("\nتعداد train:", len(en_mlm_tokenized["train"]))
print("تعداد validation:", len(en_mlm_tokenized["validation"]))

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 18000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'special_tokens_mask'],
        num_rows: 2000
    })
})

نمونه توکنایزشده انگلیسی:
{'input_ids': [0, 43722, 35154, 21325, 43975, 100, 18878, 64289, 297, 83658, 7, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'special_tokens_mask': [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]}

توکن‌های نمونه:
['<s>', '▁Living', '▁Ton', 'gues', '▁Institute', '▁for', '▁End', 'anger', 'ed', '▁Language', 's', '</s>']

تعداد train: 18000
تعداد validation: 2000


In [15]:
# این سلول Language Adapter انگلیسی را به مدل اضافه می‌کند.
# مدل پایه و Adapter فارسی ثابت می‌مانند و فقط Adapter انگلیسی آموزش‌پذیر می‌شود.

import gc
import torch
from adapters import SeqBnInvConfig

# آزادکردن اشیای مربوط به Trainer قبلی برای کاهش مصرف حافظه
if "fa_language_trainer" in globals():
    del fa_language_trainer

gc.collect()
torch.cuda.empty_cache()

# نام Language Adapter انگلیسی
EN_LANG_ADAPTER_NAME = "en_language"

# استفاده از همان معماری Language Adapter فارسی
en_language_config = SeqBnInvConfig(
    reduction_factor=2,
    non_linearity="relu",
    inv_adapter="nice",
    inv_adapter_reduction_factor=2
)

# افزودن Adapter انگلیسی، فقط در صورتی که قبلاً ساخته نشده باشد
if EN_LANG_ADAPTER_NAME not in model.adapters_config.adapters:
    model.add_adapter(
        EN_LANG_ADAPTER_NAME,
        config=en_language_config
    )

# ثابت‌کردن مدل پایه و سایر Adapterها
# فقط Adapter انگلیسی قابل‌آموزش می‌شود
model.train_adapter(EN_LANG_ADAPTER_NAME)

# فعال‌کردن Adapter انگلیسی برای عبور داده
model.set_active_adapters(EN_LANG_ADAPTER_NAME)

# فعال‌بودن Head مربوط به Masked Language Modeling
model.active_head = "default"

# انتقال مدل به GPU
model = model.to("cuda")

# محاسبه تعداد پارامترهای قابل‌آموزش
en_trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

en_total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

en_trainable_percent = (
    100 * en_trainable_params / en_total_params
)

print("Adapter فعال:", model.active_adapters)
print("Head فعال:", model.active_head)
print("پارامترهای قابل‌آموزش:", f"{en_trainable_params:,}")
print(
    "درصد پارامترهای قابل‌آموزش:",
    f"{en_trainable_percent:.4f}%"
)

print("\nLanguage Adapter انگلیسی آماده آموزش است.")

Adapter فعال: Stack[en_language]
Head فعال: default
پارامترهای قابل‌آموزش: 7,979,904
درصد پارامترهای قابل‌آموزش: 2.7174%

Language Adapter انگلیسی آماده آموزش است.


In [16]:
# این سلول تنظیمات آموزش Language Adapter انگلیسی را تعریف می‌کند
# و Trainer مخصوص آن را می‌سازد؛ هنوز آموزش شروع نمی‌شود.
# checkpointهای میانی در فضای موقت Colab ذخیره می‌شوند تا Drive پر نشود.

from pathlib import Path
from transformers import TrainingArguments
from adapters import AdapterTrainer

# مسیر موقت checkpointها در فضای Colab
EN_LANG_TRAIN_DIR = Path(
    "/content/en_language_training"
)

# تنظیمات آموزش Language Adapter انگلیسی
en_language_training_args = TrainingArguments(
    output_dir=str(EN_LANG_TRAIN_DIR),
    overwrite_output_dir=True,

    # تنظیمات اصلی آموزش
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    # ارزیابی و ذخیره در پایان هر epoch
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    # انتخاب بهترین checkpoint براساس کمترین Validation Loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # کاهش مصرف حافظه و افزایش سرعت روی Tesla T4
    fp16=True,
    weight_decay=0.01,

    # نگه‌داشتن تعداد محدود checkpoint
    save_total_limit=2,

    # غیرفعال‌کردن سرویس‌های گزارش خارجی
    report_to="none",

    # تکرارپذیری
    seed=SEED,
    data_seed=SEED
)

# ساخت Trainer مخصوص آموزش Adapter انگلیسی
en_language_trainer = AdapterTrainer(
    model=model,
    args=en_language_training_args,
    train_dataset=en_mlm_tokenized["train"],
    eval_dataset=en_mlm_tokenized["validation"],
    data_collator=mlm_data_collator,
    processing_class=tokenizer
)

# محاسبه Batch مؤثر و تعداد تقریبی گام‌ها
effective_batch_size = (
    en_language_training_args.per_device_train_batch_size
    * en_language_training_args.gradient_accumulation_steps
)

steps_per_epoch = (
    len(en_mlm_tokenized["train"])
    // effective_batch_size
)

print("Trainer انگلیسی با موفقیت ساخته شد.")
print("Batch مؤثر:", effective_batch_size)
print("گام تقریبی در هر epoch:", steps_per_epoch)
print("کل گام‌های تقریبی:", steps_per_epoch * 3)
print("مسیر موقت checkpointها:", EN_LANG_TRAIN_DIR)

Trainer انگلیسی با موفقیت ساخته شد.
Batch مؤثر: 32
گام تقریبی در هر epoch: 562
کل گام‌های تقریبی: 1686
مسیر موقت checkpointها: /content/en_language_training


In [17]:
# این سلول Language Adapter انگلیسی را با روش Masked Language Modeling آموزش می‌دهد
# و زمان آموزش، حافظه GPU و Training Loss نهایی را ثبت می‌کند.

import time
import torch

# آزادکردن حافظه‌های بلااستفاده GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# ثبت زمان شروع
en_training_start_time = time.perf_counter()

# شروع آموزش Language Adapter انگلیسی
en_language_train_result = en_language_trainer.train()

# محاسبه زمان کل آموزش
en_training_time_seconds = (
    time.perf_counter() - en_training_start_time
)

en_training_time_minutes = (
    en_training_time_seconds / 60
)

# بیشترین حافظه مصرف‌شده GPU
en_peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

print("\nآموزش Language Adapter انگلیسی تمام شد.")

print(
    "زمان آموزش:",
    f"{en_training_time_minutes:.2f} دقیقه"
)

print(
    "بیشترین حافظه GPU:",
    f"{en_peak_gpu_memory_gb:.2f} GB"
)

print(
    "Training Loss:",
    en_language_train_result.training_loss
)

Epoch,Training Loss,Validation Loss
1,5.859900,2.758673
2,5.667900,2.577301
3,5.201900,2.504606


آموزش Language Adapter انگلیسی تمام شد.
زمان آموزش: 15.02 دقیقه
بیشترین حافظه GPU: 8.15 GB
Training Loss: 5.671885190594471


In [18]:
# این سلول Adapter انگلیسی و نتیجه آموزش آن را ذخیره می‌کند.

import json
import math

EN_LANG_ADAPTER_DIR = OUTPUT_DIR / "en_language_adapter"
EN_LANGUAGE_RESULTS_FILE = RESULTS_DIR / "en_language_adapter_results.json"

# ارزیابی و ذخیره Adapter
eval_result = en_language_trainer.evaluate()
eval_loss = float(eval_result["eval_loss"])
best_loss = float(en_language_trainer.state.best_metric)

model.save_adapter(
    str(EN_LANG_ADAPTER_DIR),
    EN_LANG_ADAPTER_NAME
)

# ذخیره خلاصه نتایج
results = {
    "adapter": EN_LANG_ADAPTER_NAME,
    "train_samples": len(en_mlm_tokenized["train"]),
    "validation_samples": len(en_mlm_tokenized["validation"]),
    "epochs": 3,
    "training_loss": float(en_language_train_result.training_loss),
    "best_validation_loss": best_loss,
    "evaluation_loss": eval_loss,
    "perplexity": math.exp(eval_loss),
    "training_time_minutes": en_training_time_minutes,
    "peak_gpu_memory_gb": en_peak_gpu_memory_gb
}

with open(EN_LANGUAGE_RESULTS_FILE, "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

print("Adapter ذخیره شد:", EN_LANG_ADAPTER_DIR)
print("Best validation loss:", round(best_loss, 4))
print("Evaluation loss:", round(eval_loss, 4))
print("Perplexity:", round(math.exp(eval_loss), 4))

Adapter ذخیره شد: /content/drive/MyDrive/persian-ner-madx/outputs/en_language_adapter
Best validation loss: 2.5046
Evaluation loss: 2.5076
Perplexity: 12.2758


## 5. آموزش NER Task Adapter با داده انگلیسی


In [19]:
# این سلول داده‌های WikiANN را برای آموزش و ارزیابی NER آماده می‌کند.

MAX_NER_LENGTH = 128

def tokenize_ner(batch):
    tokens = tokenizer(
        batch["tokens"],
        truncation=True,
        max_length=MAX_NER_LENGTH,
        is_split_into_words=True
    )

    labels = []

    for i, word_labels in enumerate(batch["ner_tags"]):
        word_ids = tokens.word_ids(batch_index=i)
        previous_word = None
        aligned_labels = []

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word:
                aligned_labels.append(word_labels[word_id])
            else:
                aligned_labels.append(-100)

            previous_word = word_id

        labels.append(aligned_labels)

    tokens["labels"] = labels
    return tokens


en_ner_tokenized = english_dataset.map(
    tokenize_ner,
    batched=True,
    remove_columns=english_dataset["train"].column_names
)

fa_ner_tokenized = persian_dataset.map(
    tokenize_ner,
    batched=True,
    remove_columns=persian_dataset["train"].column_names
)

print(en_ner_tokenized)
print("\nنمونه input_ids:", en_ner_tokenized["train"][0]["input_ids"])
print("نمونه labels:", en_ner_tokenized["train"][0]["labels"])

#داده‌های NER انگلیسی و فارسی را توکن‌سازی می‌کنیم و برچسب هر کلمه را فقط به اولین زیرتوکن می‌دهیم.

DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})

نمونه input_ids: [0, 627, 5, 841, 5, 947, 24658, 7, 15, 2907, 5, 155484, 32547, 1388, 15, 483, 16028, 46298, 1388, 2]
نمونه labels: [-100, 3, -100, -100, -100, 4, -100, -100, 0, 3, -100, 4, 4, 0, 0, 0, -100, 0, 0, -100]


In [20]:
# این سلول مدل را برای آموزش NER آماده می‌کند.
# Adapter انگلیسی برای زبان و Adapter جدید برای تشخیص موجودیت استفاده می‌شود.

import gc
import torch
from adapters import AutoAdapterModel, SeqBnConfig, Stack

# خالی‌کردن حافظه مدل قبلی
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

# بارگذاری مدل پایه
task_model = AutoAdapterModel.from_pretrained(MODEL_NAME)

# بارگذاری Language Adapterهای آموزش‌دیده
task_model.load_adapter(
    str(EN_LANG_ADAPTER_DIR),
    load_as=EN_LANG_ADAPTER_NAME
)

task_model.load_adapter(
    str(LANG_ADAPTER_DIR),
    load_as=LANG_ADAPTER_NAME
)

# اضافه‌کردن Task Adapter و خروجی NER
TASK_ADAPTER_NAME = "ner_task"

task_model.add_adapter(
    TASK_ADAPTER_NAME,
    config=SeqBnConfig(reduction_factor=16)
)

task_model.add_tagging_head(
    TASK_ADAPTER_NAME,
    num_labels=num_labels,
    id2label=id2label
)

# فقط Task Adapter و Head آموزش داده می‌شوند
task_model.train_adapter(TASK_ADAPTER_NAME)

task_model.set_active_adapters(
    Stack(EN_LANG_ADAPTER_NAME, TASK_ADAPTER_NAME)
)

task_model.active_head = TASK_ADAPTER_NAME
task_model.to("cuda")

# تعداد پارامترهای قابل‌آموزش
trainable_params = sum(
    p.numel() for p in task_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in task_model.parameters()
)

print("Adapter فعال:", task_model.active_adapters)
print("Head فعال:", task_model.active_head)
print("پارامتر قابل‌آموزش:", f"{trainable_params:,}")
print("درصد قابل‌آموزش:", f"{100 * trainable_params / total_params:.4f}%")

Adapter فعال: Stack[en_language, ner_task]
Head فعال: ner_task
پارامتر قابل‌آموزش: 1,742,041
درصد قابل‌آموزش: 0.5914%


In [21]:
# این سلول داده‌ها را برای batch‌بندی آماده می‌کند
# و معیارهای Precision، Recall، F1 و Accuracy را می‌سازد.

import numpy as np
import evaluate
from transformers import DataCollatorForTokenClassification

data_collator_ner = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

seqeval = evaluate.load("seqeval")

def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = [
        [label_names[p] for p, label in zip(pred, row) if label != -100]
        for pred, row in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[label] for label in row if label != -100]
        for row in labels
    ]

    result = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": result["overall_precision"],
        "recall": result["overall_recall"],
        "f1": result["overall_f1"],
        "accuracy": result["overall_accuracy"]
    }

print("معیارهای ارزیابی NER آماده شدند.")

معیارهای ارزیابی NER آماده شدند.


In [22]:
# این سلول تنظیمات آموزش Task Adapter مربوط به NER را آماده می‌کند.

from transformers import TrainingArguments
from adapters import AdapterTrainer

NER_TRAIN_DIR = "/content/ner_task_training"

ner_training_args = TrainingArguments(
    output_dir=NER_TRAIN_DIR,
    num_train_epochs=5,
    learning_rate=1e-4,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=True,
    weight_decay=0.01,
    save_total_limit=2,
    report_to="none",
    seed=SEED
)

ner_trainer = AdapterTrainer(
    model=task_model,
    args=ner_training_args,
    train_dataset=en_ner_tokenized["train"],
    eval_dataset=en_ner_tokenized["validation"],
    data_collator=data_collator_ner,
    compute_metrics=compute_ner_metrics,
    processing_class=tokenizer
)

print("Trainer آماده شد.")
print("Batch مؤثر:", 16 * 2)
print("تعداد epoch:", 5)
print("مسیر checkpointها:", NER_TRAIN_DIR)

Trainer آماده شد.
Batch مؤثر: 32
تعداد epoch: 5
مسیر checkpointها: /content/ner_task_training


In [23]:
# این سلول Task Adapter مربوط به NER را آموزش می‌دهد
# و زمان آموزش و مصرف حافظه GPU را ثبت می‌کند.

import time
import torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

ner_train_result = ner_trainer.train()

train_time_minutes = (
    time.perf_counter() - start_time
) / 60

peak_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

print("\nآموزش NER تمام شد.")
print("زمان آموزش:", round(train_time_minutes, 2), "دقیقه")
print("بیشترین حافظه GPU:", round(peak_memory_gb, 2), "GB")
print("Training loss:", ner_train_result.training_loss)
print("Best F1:", ner_trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.776200,0.353326,0.671999,0.735034,0.702105,0.888842
2,0.673300,0.316238,0.722753,0.766277,0.743879,0.901171
3,0.602200,0.301698,0.739655,0.781367,0.759939,0.906366
4,0.572500,0.295466,0.743761,0.781226,0.762033,0.908305
5,0.558900,0.293388,0.750999,0.785264,0.767750,0.909324


آموزش NER تمام شد.
زمان آموزش: 10.71 دقیقه
بیشترین حافظه GPU: 2.0 GB
Training loss: 0.7490592474365234
Best F1: 0.7677495324513403


In [24]:
# این سلول Task Adapter آموزش‌دیده و NER Head را روی Google Drive ذخیره می‌کند.

from pathlib import Path
import shutil

TASK_ADAPTER_DIR = Path(
    "/content/drive/MyDrive/persian-ner-madx/outputs/en_ner_task_adapter"
)

# پوشه خالی یا ناقص قبلی پاک می‌شود.
if TASK_ADAPTER_DIR.exists():
    shutil.rmtree(TASK_ADAPTER_DIR)

TASK_ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)

task_model.save_adapter(
    str(TASK_ADAPTER_DIR),
    "ner_task",
    with_head=True
)

print("Task Adapter ذخیره شد:\n")

for file in TASK_ADAPTER_DIR.rglob("*"):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"- {file.relative_to(TASK_ADAPTER_DIR)} | {size_mb:.2f} MB")

Task Adapter ذخیره شد:

- adapter_config.json | 0.00 MB
- pytorch_adapter.bin | 3.43 MB
- head_config.json | 0.00 MB
- pytorch_model_head.bin | 0.02 MB


## 6. ارزیابی نهایی انگلیسی و فارسی Zero-shot


In [25]:
# این سلول مدل را روی تست انگلیسی و سپس به‌صورت Zero-shot روی تست فارسی
# ارزیابی می‌کند و نتایج نهایی را در Google Drive ذخیره می‌کند.

import json
from adapters import Stack

def select_metrics(metrics):
    wanted = ["eval_precision", "eval_recall", "eval_f1", "eval_accuracy"]
    return {
        key.replace("eval_", ""): float(metrics[key])
        for key in wanted
    }


# ارزیابی روی زبان مبدأ: انگلیسی
task_model.set_active_adapters(
    Stack(EN_LANG_ADAPTER_NAME, TASK_ADAPTER_NAME)
)
task_model.active_head = TASK_ADAPTER_NAME

english_test_metrics = ner_trainer.evaluate(
    en_ner_tokenized["test"]
)

english_results = select_metrics(english_test_metrics)


# ارزیابی Zero-shot روی زبان هدف: فارسی
task_model.set_active_adapters(
    Stack(LANG_ADAPTER_NAME, TASK_ADAPTER_NAME)
)
task_model.active_head = TASK_ADAPTER_NAME

persian_test_metrics = ner_trainer.evaluate(
    fa_ner_tokenized["test"]
)

persian_results = select_metrics(persian_test_metrics)


# ذخیره نتایج نهایی
madx_results = {
    "english_source_test": english_results,
    "persian_zero_shot_test": persian_results,
    "best_english_validation_f1": float(ner_trainer.state.best_metric)
}

result_path = RESULTS_DIR / "madx_zero_shot_results.json"

with open(result_path, "w", encoding="utf-8") as file:
    json.dump(madx_results, file, ensure_ascii=False, indent=2)


print("\nنتایج تست انگلیسی:")
for name, value in english_results.items():
    print(f"{name}: {value:.4f}")

print("\nنتایج Zero-shot فارسی:")
for name, value in persian_results.items():
    print(f"{name}: {value:.4f}")

print("\nنتایج ذخیره شد در:")
print(result_path)

نتایج تست انگلیسی:
precision: 0.7509
recall: 0.7872
f1: 0.7686
accuracy: 0.9115

نتایج Zero-shot فارسی:
precision: 0.7412
recall: 0.8093
f1: 0.7738
accuracy: 0.8891

نتایج ذخیره شد در:
/content/drive/MyDrive/persian-ner-madx/results/madx_zero_shot_results.json


In [26]:
# این سلول نتایج انگلیسی و فارسی Zero-shot را در یک فایل CSV جدا ذخیره می‌کند.

import pandas as pd

comparison_df = pd.DataFrame([
    {
        "setting": "English source test",
        "language_adapter": "English",
        "task_training_language": "English",
        "precision": english_results["precision"],
        "recall": english_results["recall"],
        "f1": english_results["f1"],
        "accuracy": english_results["accuracy"]
    },
    {
        "setting": "Persian zero-shot test",
        "language_adapter": "Persian",
        "task_training_language": "English",
        "precision": persian_results["precision"],
        "recall": persian_results["recall"],
        "f1": persian_results["f1"],
        "accuracy": persian_results["accuracy"]
    }
])

csv_path = RESULTS_DIR / "madx_comparison.csv"
comparison_df.to_csv(csv_path, index=False)

display(comparison_df)

print("\nفایل جدول ذخیره شد:")
print(csv_path)

,setting,language_adapter,task_training_language,precision,recall,f1,accuracy
0,English source test,English,English,0.750910,0.787174,0.768614,0.911498
1,Persian zero-shot test,Persian,English,0.741176,0.809346,0.773763,0.889109


فایل جدول ذخیره شد:
/content/drive/MyDrive/persian-ner-madx/results/madx_comparison.csv


## تحلیل نتایج MAD-X

در این آزمایش، Task Adapter مربوط به تشخیص موجودیت نامدار فقط با داده‌های برچسب‌دار انگلیسی WikiANN آموزش داده شد. سپس برای ارزیابی انگلیسی، ترکیب زیر استفاده شد:

English Language Adapter → NER Task Adapter

برای ارزیابی Zero-shot فارسی، بدون آموزش Task Adapter با هیچ داده برچسب‌دار فارسی، فقط Language Adapter انگلیسی با Language Adapter فارسی جایگزین شد:

Persian Language Adapter → English-trained NER Task Adapter

نتایج نشان دادند که مدل روی داده تست انگلیسی به F1 برابر با 0.7686 و روی داده تست فارسی در حالت Zero-shot به F1 برابر با 0.7738 رسید. بنابراین Task Adapter توانسته دانش وظیفه NER را که از زبان انگلیسی یاد گرفته بود، با استفاده از Language Adapter فارسی به زبان هدف منتقل کند.

Recall مدل روی فارسی برابر با 0.8093 است که از Recall انگلیسی بیشتر است. این موضوع نشان می‌دهد مدل درصد بیشتری از موجودیت‌های فارسی را پیدا کرده است. بااین‌حال Precision فارسی کمی کمتر است؛ یعنی مدل در زبان فارسی تعدادی پیش‌بینی مثبت اشتباه بیشتری داشته است.

با وجود بالاتر بودن جزئی F1 فارسی، نمی‌توان نتیجه گرفت که مدل ذاتاً روی فارسی بهتر از انگلیسی عمل می‌کند؛ زیرا مجموعه‌های تست فارسی و انگلیسی شامل نمونه‌های یکسان نیستند و ممکن است از نظر دشواری، طول جمله و توزیع موجودیت‌ها متفاوت باشند.

این نتایج را نباید مستقیماً با نتایج Full Fine-tuning، LoRA و Adapter پروژه اصلی روی دیتاست ARMAN مقایسه کرد؛ زیرا آزمایش اصلی روی ARMAN و آزمایش MAD-X روی WikiANN انجام شده‌اند و نوع برچسب‌ها و توزیع داده‌های آن‌ها متفاوت است.

### محدودیت آزمایش

این پیاده‌سازی یک نسخه سبک و آموزشی از MAD-X است. Language Adapterهای فارسی و انگلیسی فقط با ۱۸ هزار جمله کوتاه WikiANN آموزش داده شدند، درحالی‌که در نسخه اصلی MAD-X معمولاً از حجم بسیار بیشتری از متن‌های تک‌زبانه، مانند داده‌های Wikipedia، برای آموزش Language Adapter استفاده می‌شود. بنابراین استفاده از داده بیشتر و متنوع‌تر می‌تواند کیفیت انتقال بین‌زبانی را بهتر کند.

### نتیجه‌گیری

نتایج نشان می‌دهند که جداسازی دانش زبان و دانش وظیفه با استفاده از Language Adapter و Task Adapter، امکان اجرای NER فارسی به‌صورت Cross-lingual Zero-shot را فراهم کرده است. در این روش هیچ داده برچسب‌دار فارسی برای آموزش Task Adapter استفاده نشد و مدل پایه XLM-R نیز ثابت باقی ماند.

## معماری نهایی پروژه

در این پروژه از مدل چندزبانه `xlm-roberta-base` به‌عنوان مدل پایه استفاده شد. پارامترهای اصلی مدل در تمام مراحل ثابت باقی ماندند. در مراحل سازگارسازی زبانی فقط Adapterهای کوچک و در مرحله NER فقط Task Adapter و NER Head آموزش داده شدند.

معماری MAD-X استفاده‌شده به‌صورت زیر است:

### آموزش Language Adapter فارسی

متن‌های فارسی بدون برچسب WikiANN با هدف Masked Language Modeling استفاده شدند:

Persian text → XLM-R + Persian Language Adapter

در این مرحله فقط `fa_language` آموزش داده شد.

### آموزش Language Adapter انگلیسی

متن‌های انگلیسی بدون برچسب WikiANN با همان هدف Masked Language Modeling استفاده شدند:

English text → XLM-R + English Language Adapter

در این مرحله فقط `en_language` آموزش داده شد.

### آموزش Task Adapter

برای یادگیری تسک NER، داده‌های برچسب‌دار انگلیسی WikiANN استفاده شدند:

English Language Adapter → NER Task Adapter → NER Head

در این مرحله مدل پایه و Language Adapterها ثابت بودند و فقط `ner_task` و NER Head آموزش داده شدند.

### انتقال Zero-shot به فارسی

برای اجرای NER روی فارسی، Task Adapter آموزش‌دیده با انگلیسی بدون تغییر نگه داشته شد و فقط Language Adapter انگلیسی با Adapter فارسی جایگزین شد:

Persian Language Adapter → English-trained NER Task Adapter → NER Head

بنابراین هیچ داده برچسب‌دار فارسی در آموزش Task Adapter استفاده نشد.

## فایل‌های خروجی

فایل‌های اصلی پروژه در Google Drive ذخیره شده‌اند:

- `outputs/fa_language_adapter`  
  Language Adapter آموزش‌دیده فارسی

- `outputs/en_language_adapter`  
  Language Adapter آموزش‌دیده انگلیسی

- `outputs/en_ner_task_adapter`  
  Task Adapter مربوط به NER به همراه NER Head

- `results/fa_language_adapter_results.json`  
  نتایج آموزش Language Adapter فارسی

- `results/en_language_adapter_results.json`  
  نتایج آموزش Language Adapter انگلیسی

- `results/madx_zero_shot_results.json`  
  نتایج تست انگلیسی و ارزیابی Zero-shot فارسی

- `results/madx_comparison.csv`  
  جدول نهایی مقایسه نتایج انگلیسی و فارسی

## جمع‌بندی نهایی نتایج

| حالت ارزیابی | Precision | Recall | F1 | Accuracy |
|---|---:|---:|---:|---:|
| English source test | 0.7509 | 0.7872 | 0.7686 | 0.9115 |
| Persian zero-shot test | 0.7412 | 0.8093 | 0.7738 | 0.8891 |

این آزمایش نشان داد که MAD-X می‌تواند دانش وظیفه NER را از زبان انگلیسی به فارسی منتقل کند، درحالی‌که مدل پایه ثابت مانده و هیچ نمونه برچسب‌دار فارسی برای آموزش Task Adapter استفاده نشده است.